# Import libraries

In [26]:
import os
import logging
from typing import Any
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.output_parsers import BaseOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

# Connect to ChromaDB

In [9]:
api_key = os.getenv("API_KEY")
llm = model = ChatGoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.3,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
namespace = f"chroma/{collection_name}"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

# Initialize retriever

In [14]:
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm,
)

In [18]:
retriever.invoke(
    "What are Morphemes?"
)

[Document(id='973f1fde-483e-5c55-8ec2-f3584100f529', metadata={'total_pages': 34, 'contains_code_or_cli': False, 'talks_about_ngrams': False, 'talks_about_edit_distance': False, 'talks_about_evaluation_perplexity': False, 'linguistic_focus': 'English', 'page': 3, 'talks_about_morphology': True, 'talks_about_language_modeling': True, 'chapter_number': 2, 'importance_score': 'High', 'talks_about_unicode': False, 'talks_about_smoothing_interpolation': False, 'content_type': 'Narrative', 'talks_about_bpe': False, 'creationdate': 'D:20260329110007', 'contains_table': False, 'contains_regex': False, 'source': 'data\\book_chapter_02.pdf', 'contains_math_latex': True, 'volume': 'II: Annotating Linguistic Structure', 'talks_about_tokenization': False, 'section_title': '2.1 • W ORDS'}, page_content="2.1 • W ORDS 7\nbecause the relationship between the number of types |V | and number of instances\nN is called Herdan’s Law (Herdan, 1960) or Heaps’ Law (Heaps, 1978) after itsHerdan’s Law\nHeaps’ La

# Creation of custom questions

In [25]:
class LineList(BaseModel):
    # "lines" is the key (attribute name) of the parsed output
    lines: list[str] = Field(description="Lines of text")

class LineListOutputParser(PydanticOutputParser[Any]):
    def __init__(self) -> None:
        super().__init__(pydantic_object=LineList)

    def parse(self, text: str) -> LineList:
        lines = text.strip().splitlines()
        return LineList(lines=lines)

## Creation of custom prompt

In [20]:
# Definición de esquema de salida de preguntas
class LineListOutputParser(BaseOutputParser):
    def parse(self, text: str):
        # Splits the output by newline and removes empty lines
        return [line.strip() for line in text.strip().split("\n") if line.strip()]

In [21]:
# Creación de prompt personalizado
prompt = PromptTemplate.from_template(
    """You are an AI language assistant well versed in LLMs.
Your more precise task is to generate five different versions of the given question to retrieve relevant documents from a vector database.
By generating multiple perspectives on the question, your goal is to overcome some of the limitations of the distance-based similarity search.

Provide these alternative questions separed by newlines.

Original question: {question}
New questions:"""
)

# In language expression language, you could create the chain with:
llm_chain = prompt | llm | LineListOutputParser()
llm_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language assistant well versed in LLMs.\nYour more precise task is to generate five different versions of the given question to retrieve relevant documents from a vector database.\nBy generating multiple perspectives on the question, your goal is to overcome some of the limitations of the distance-based similarity search.\n\nProvide these alternative questions separed by newlines.\n\nOriginal question: {question}\nNew questions:')
| ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=

In [22]:
# Use de cadena de generación de preguntas personalizada
llm_chain.invoke(
    {"question": "What are Morphemes?"}
)

['What is the definition of a morpheme?',
 'Explain the concept of morphemes in linguistics.',
 'Describe the smallest meaningful units of language, known as morphemes.',
 'What are the fundamental building blocks of words in terms of their meaning?',
 'Can you provide an overview of what constitutes a morpheme?']

In [23]:
# Integración de cadena de generación de preguntas personalizada en retriever
retriever = MultiQueryRetriever(
    retriever=vectorstore.as_retriever(),
    llm_chain=llm_chain,
    parser_key="lines",
)

In [27]:
# Uso de retriever con cadena de generación de preguntas personalizada
retriever.invoke(
    "What are Morphemes?"
)

[Document(id='973f1fde-483e-5c55-8ec2-f3584100f529', metadata={'talks_about_language_modeling': True, 'linguistic_focus': 'English', 'page': 3, 'talks_about_evaluation_perplexity': False, 'contains_regex': False, 'source': 'data\\book_chapter_02.pdf', 'creationdate': 'D:20260329110007', 'talks_about_edit_distance': False, 'content_type': 'Narrative', 'volume': 'II: Annotating Linguistic Structure', 'talks_about_smoothing_interpolation': False, 'contains_table': False, 'contains_code_or_cli': False, 'talks_about_bpe': False, 'importance_score': 'High', 'section_title': '2.1 • W ORDS', 'talks_about_tokenization': False, 'talks_about_ngrams': False, 'total_pages': 34, 'contains_math_latex': True, 'talks_about_unicode': False, 'chapter_number': 2, 'talks_about_morphology': True}, page_content="2.1 • W ORDS 7\nbecause the relationship between the number of types |V | and number of instances\nN is called Herdan’s Law (Herdan, 1960) or Heaps’ Law (Heaps, 1978) after itsHerdan’s Law\nHeaps’ La